# Baymax — Module 7: food recognizer → TensorFlow Lite (accuracy-first)

Trains an image classifier for the **13 catalogue foods** in the cleaned Food-101 subset and exports a **float16 TFLite model**
for the Flutter app (float16 keeps the accuracy of the trained model; no int8 shortcut).

**How to use on Kaggle**
1. *Add Input* → your uploaded **food dataset** (the zip with `images/`, `split.csv`, `labels.json`).
   **Optional but needed for Pakistani dishes:** also add your own photos as a second input — a dataset containing a folder
   `custom_photos/<catalogue_id>/<session>/*.jpg` (see `PAKISTANI_PHOTOS.md` / `pakistani_photos_template.zip`).
2. Right sidebar: **Accelerator = GPU T4 x2** (or P100) and **Internet = On** (needed once to download the ImageNet starting weights; Kaggle asks for phone verification).
3. *Run All* (about 30–60 minutes). Download everything from `/kaggle/working`.

**What makes it accurate:** a strong pretrained backbone (EfficientNetV2-B0), two-stage fine-tuning, augmentation, label smoothing, an automatic
**label-noise pass** (Food-101 training photos contain mislabeled pictures), a **held-out test set** that is never used for any decision,
and a **shipping gate** — the model file is only written under its final name if it passes accuracy, per-class and TFLite-parity checks.

In [ ]:
import os, json, glob, time, random, warnings
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
try:
    display
except NameError:
    display = print

QUICK = os.environ.get('FOOD_QUICK') == '1'            # tiny smoke-test mode (CPU, no pretrained weights)
INPUT_DIR = os.environ.get('FOOD_INPUT_DIR', '/kaggle/input')
OUT_DIR = os.environ.get('FOOD_OUT_DIR', '/kaggle/working')
BACKBONE = os.environ.get('FOOD_BACKBONE', 'EfficientNetV2B0')   # EfficientNetV2B0 | MobileNetV3Large | MobileNetV2
IMG = 128 if QUICK else 256
BATCH = 16 if QUICK else 32
SEED = 7
EPOCHS_HEAD, EPOCHS_FT, EPOCHS_CLEAN = (1, 1, 1) if QUICK else (8, 22, 6)
WEIGHTS = None if QUICK else 'imagenet'
AUTOTUNE = tf.data.AUTOTUNE

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
print('backbone:', BACKBONE, '| image size:', IMG, '| quick mode:', QUICK)

## 1a. Your own photos (optional — this is how Pakistani dishes get in)
If a second input contains a folder `custom_photos/<catalogue_id>/<session>/*.jpg`, those photos are cleaned (rotation fixed, duplicates removed) and
**split by session** — all photos of one plate/meal go to the same side — into train / selection / test. A dish is trained **only if it has enough photos
to be tested fairly** (100+ usable photos from at least ~6 different sessions, so that 15+ test photos exist); otherwise it is skipped and the app keeps manual selection for it.
The final shipping gate below applies to these dishes exactly as to the Food-101 ones.

In [ ]:
import hashlib, difflib
from PIL import Image, ImageOps

VALID_IDS = set(["boiled_egg", "fried_egg", "omelette", "oatmeal", "cornflakes", "paratha", "aloo_paratha", "roti", "naan", "white_bread", "wholewheat_bread", "halwa_puri", "khichdi", "milk", "skim_milk", "yogurt", "raita", "lassi_sweet", "lassi_salted", "paneer", "cheese_slice", "tea_milk", "green_tea", "black_coffee", "white_rice", "brown_rice", "quinoa", "pasta", "pulao", "chicken_biryani", "potato_boiled", "sweet_potato", "french_fries", "grilled_chicken", "chicken_karahi", "chicken_curry", "butter_chicken", "chicken_tikka", "seekh_kebab", "shami_kebab", "beef_nihari", "mutton_curry", "haleem", "grilled_fish", "fish_curry", "tuna_canned", "grilled_shrimp", "prawn_curry", "daal", "chana_masala", "rajma", "mixed_veg_curry", "aloo_gobi", "palak_paneer", "bhindi", "steamed_veg", "green_salad", "chicken_salad", "chicken_soup", "hummus", "tofu", "samosa", "pakora", "chicken_burger", "chicken_sandwich", "pizza_slice", "instant_noodles", "potato_chips", "biscuits", "cake_slice", "chocolate", "ice_cream", "gulab_jamun", "kheer", "suji_halwa", "almonds", "mixed_nuts", "peanuts", "peanut_butter", "butter", "ghee", "olive_oil", "dates", "banana", "apple", "orange", "mango", "guava", "grapes", "watermelon", "papaya", "strawberries", "pomegranate", "orange_juice", "cola", "protein_shake", "coconut_water"])
CUSTOM_MIN_IMAGES, CUSTOM_MIN_TRAIN, CUSTOM_MIN_SELECT, CUSTOM_MIN_TEST = 100, 50, 15, 15
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

def _dhash(im, size=8):
    g = np.asarray(im.convert('L').resize((size + 1, size), Image.LANCZOS), dtype=np.int16)
    return int(''.join('1' if b else '0' for b in (g[:, :-1] > g[:, 1:]).ravel()), 2)

def load_custom_photos():
    """Reads  custom_photos/<catalogue_id>/[<session>/]*.jpg , cleans it and splits it BY SESSION.
    Photos of the same plate/meal must stay together, otherwise near-identical shots land in both train and test
    and the accuracy is fake. Returns (rows DataFrame, report dict)."""
    cols = ['path', 'label', 'split', 'source_class', 'md5', 'grp', 'half']
    roots = [d for d in glob.glob(os.path.join(INPUT_DIR, '**', 'custom_photos'), recursive=True) if os.path.isdir(d)]
    if not roots:
        print('No custom_photos folder found -> training on the Food-101 foods only (no Pakistani dishes).')
        return pd.DataFrame(columns=cols), {}
    root = roots[0]
    out_dir = os.path.join(OUT_DIR, 'custom_clean'); os.makedirs(out_dir, exist_ok=True)
    rows, report, seen_md5 = [], {}, {}
    for food in sorted(f for f in os.listdir(root) if os.path.isdir(os.path.join(root, f))):
        if food not in VALID_IDS:
            near = difflib.get_close_matches(food, sorted(VALID_IDS), n=3)
            raise SystemExit(f'custom_photos/{food}: not a catalogue food id. Did you mean {near}? '
                             f'(ids must match assets/data/foods.json exactly)')
        files = []
        for dp, _, fns in os.walk(os.path.join(root, food)):
            for fn in sorted(fns):
                if fn.startswith('.'):
                    continue
                files.append(os.path.join(dp, fn))
        rep = {'found': len(files), 'unreadable': 0, 'too_small_or_odd': 0, 'duplicates': 0, 'heic': 0, 'sessions': 0, 'ungrouped': False}
        kept = []   # (grp, md5, dhash, pil_image)
        for p in files:
            if p.lower().endswith(('.heic', '.heif')):
                rep['heic'] += 1; continue
            if not p.lower().endswith(EXTS):
                continue
            try:
                with open(p, 'rb') as fh:
                    md5 = hashlib.md5(fh.read()).hexdigest()
                with Image.open(p) as im:
                    im = ImageOps.exif_transpose(im).convert('RGB')     # phone photos carry a rotation flag
                    im.load()
            except Exception:
                rep['unreadable'] += 1; continue
            w, h = im.size
            if min(w, h) < 96 or max(w, h) / min(w, h) > 4:
                rep['too_small_or_odd'] += 1; continue
            if md5 in seen_md5:
                rep['duplicates'] += 1; continue
            seen_md5[md5] = p
            rel = os.path.relpath(p, os.path.join(root, food)).split(os.sep)
            grp = rel[0] if len(rel) > 1 else f'single:{md5}'            # sub-folder = one session/plate/meal
            im.thumbnail((384, 384), Image.LANCZOS)
            kept.append((grp, md5, _dhash(im), im))
        # near-identical pictures taken in DIFFERENT sessions would leak between splits -> keep only one
        keep2, hashes = [], []
        for grp, md5, dh, im in kept:
            if any(bin(dh ^ o).count('1') <= 3 and g2 != grp for o, g2 in hashes):
                rep['duplicates'] += 1; continue
            hashes.append((dh, grp)); keep2.append((grp, md5, dh, im))
        kept = keep2
        groups = sorted({k[0] for k in kept}, key=lambda g: hashlib.md5(f'{food}/{g}'.encode()).hexdigest())
        rep['sessions'] = len(groups); rep['ungrouped'] = any(g.startswith('single:') for g in groups)
        n = len(kept); rep['usable'] = n
        # assign whole sessions: ~60% train / 20% selection / 20% test, in a fixed pseudo-random order
        size = Counter(k[0] for k in kept); assign, cum = {}, 0
        for g in groups:
            assign[g] = 'train' if cum < 0.60 * n else ('select' if cum < 0.80 * n else 'test')
            cum += size[g]
        cnt = Counter(assign[k[0]] for k in kept)
        rep.update(train=cnt['train'], select=cnt['select'], test=cnt['test'])
        problems = []
        if n < CUSTOM_MIN_IMAGES: problems.append(f'only {n} usable photos (need {CUSTOM_MIN_IMAGES}+)')
        if cnt['train'] < CUSTOM_MIN_TRAIN: problems.append(f"{cnt['train']} train photos (need {CUSTOM_MIN_TRAIN}+)")
        if cnt['select'] < CUSTOM_MIN_SELECT or cnt['test'] < CUSTOM_MIN_TEST:
            problems.append(f"selection/test would have {cnt['select']}/{cnt['test']} photos (need {CUSTOM_MIN_SELECT}+/{CUSTOM_MIN_TEST}+; "
                            f"more sessions of this dish help, got {len(groups)})")
        rep['kept'] = not problems; rep['problems'] = problems
        report[food] = rep
        if problems:
            continue      # not enough photos to measure accuracy honestly -> the dish is NOT trained; the app keeps manual pick
        os.makedirs(os.path.join(out_dir, food), exist_ok=True)
        for k, (grp, md5, dh, im) in enumerate(kept):
            path = os.path.join(out_dir, food, f'{k:04d}.jpg')
            im.save(path, 'JPEG', quality=90)
            a = assign[grp]
            rows.append({'path': path, 'label': food, 'split': 'train' if a == 'train' else 'val', 'source_class': 'custom:' + grp,
                         'md5': md5, 'grp': grp, 'half': '' if a == 'train' else a})
    rep_df = pd.DataFrame(report).T
    if len(rep_df):
        display(rep_df[['found', 'usable', 'sessions', 'train', 'select', 'test', 'duplicates', 'unreadable', 'too_small_or_odd', 'heic', 'kept']])
    for food, r in report.items():
        if not r['kept']:
            print(f'  NOT TRAINED  {food}: ' + '; '.join(r['problems']))
        elif r['ungrouped']:
            print(f'  WARNING {food}: photos are not in session sub-folders, so shots of the same plate can fall in both train and test '
                  f'and its accuracy may look better than it is.')
        if r['heic']:
            print(f'  {food}: {r["heic"]} HEIC photos skipped - convert them to JPG (iPhone: Settings > Camera > Formats > Most Compatible).')
    json.dump(report, open(os.path.join(OUT_DIR, 'custom_photos_report.json'), 'w'), indent=1)
    return pd.DataFrame(rows, columns=cols), report

## 1b. Load the cleaned dataset
`split.csv` holds a fixed train/validation split (validation = Food-101's manually cleaned official test images).
The validation images are split again: **half for choosing between options (selection set)** and
**half held out as the final test set** that no decision ever sees, so the reported accuracy is honest.
(Food-101 images are halved by file hash; your own photos by session, see 1a.)

In [ ]:
split_files = glob.glob(os.path.join(INPUT_DIR, '**', 'split.csv'), recursive=True)
if not split_files:
    raise SystemExit('split.csv not found. Use Add Input -> your food dataset (the zip containing images/ and split.csv).')
ROOT = os.path.dirname(split_files[0])
meta = pd.read_csv(split_files[0])
meta['path'] = meta['path'].apply(lambda p: os.path.join(ROOT, p))
for col, default in (('grp', ''), ('half', ''), ('source', 'food101')):      # the final package already carries these columns
    if col not in meta.columns:
        meta[col] = default
meta[['grp', 'half', 'source']] = meta[['grp', 'half', 'source']].fillna('')
custom_df, custom_report = load_custom_photos()
if len(custom_df):
    custom_df['source'] = 'team_photos'
meta = pd.concat([meta, custom_df], ignore_index=True)
labels = sorted(meta['label'].unique())
label_to_id = {l: i for i, l in enumerate(labels)}
meta['y'] = meta['label'].map(label_to_id)

train_df = meta[meta.split == 'train'].reset_index(drop=True)
val_all = meta[meta.split == 'val'].reset_index(drop=True)
# Food-101 images: deterministic 50/50 split of the validation images by file hash.
# Web-scraped Pakistani images and your own photos: halves were assigned per unique picture / per session (never by single file).
by_hash = val_all['md5'].apply(lambda h: int(h[-1], 16) % 2 == 0)
is_test = np.where(val_all['half'] != '', val_all['half'] == 'test', by_hash)
select_df, test_df = val_all[~is_test].reset_index(drop=True), val_all[is_test].reset_index(drop=True)

if QUICK:   # smoke test: a handful of images per class
    cap = lambda d, n: d.groupby('label', group_keys=False).apply(lambda g: g.head(n)).reset_index(drop=True)
    train_df, select_df, test_df = cap(train_df, 20), cap(select_df, 8), cap(test_df, 8)

print('dataset folder:', ROOT)
print(f'{len(labels)} classes | train {len(train_df)} | selection {len(select_df)} | test {len(test_df)}')
print('image sources:', dict(meta['source'].value_counts()))
display(pd.DataFrame({'train': train_df.label.value_counts(), 'selection': select_df.label.value_counts(),
                      'test': test_df.label.value_counts()}).fillna(0).astype(int).sort_index())
assert not (set(train_df.md5) & set(test_df.md5)), 'identical files in train and test!'
assert not (set(train_df.md5) & set(select_df.md5)), 'identical files in train and selection!'

## 2. Input pipeline
Images are squashed to a square with **area (average) resizing** — the same method the Flutter app uses — so training and the phone see identical pixels.
The model takes raw 0–255 RGB; the backbone's own preprocessing is inside the model.

In [ ]:
def make_ds(df, training):
    paths, ys = df['path'].values, df['y'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    def load(p, y):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(img, (IMG, IMG), method='area')
        return tf.cast(tf.round(img), tf.uint8), y          # uint8 keeps the cache small
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

def class_weights(df):
    counts = Counter(df['y'])
    return {i: len(df) / (len(labels) * counts[i]) for i in range(len(labels))}

val_ds = make_ds(select_df, False)
test_ds = make_ds(test_df, False)

## 3. Model

In [ ]:
def build_model():
    augment = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.2),
        tf.keras.layers.RandomTranslation(0.08, 0.08),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
    ], name='augment')
    kw = dict(include_top=False, weights=WEIGHTS, input_shape=(IMG, IMG, 3))
    if WEIGHTS == 'imagenet':
        try:
            base = getattr(tf.keras.applications, BACKBONE)(**kw, **({} if BACKBONE == 'MobileNetV2' else {'include_preprocessing': True}))
        except Exception as e:
            raise SystemExit('Could not download the ImageNet weights. Turn Internet ON in the notebook settings and re-run. (%s)' % e)
    else:
        base = getattr(tf.keras.applications, BACKBONE)(**kw, **({} if BACKBONE == 'MobileNetV2' else {'include_preprocessing': True}))
    base.trainable = False
    inputs = tf.keras.Input((IMG, IMG, 3), name='image_rgb_0_255')
    x = augment(inputs)
    if BACKBONE == 'MobileNetV2':
        x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1, name='to_minus1_1')(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(labels), activation='softmax', name='probabilities', dtype='float32')(x)
    return tf.keras.Model(inputs, outputs), base

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),      # label smoothing is applied below
                  metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(3, name='top3')])

model, base = build_model()
print('parameters: %.1f M' % (model.count_params() / 1e6))

## 4. Train — stage 1 (new head) then stage 2 (fine-tune the backbone)

In [ ]:
train_ds = make_ds(train_df, True)
cw = class_weights(train_df)
callbacks = lambda: [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
                     tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)]
histories = []

compile_model(model, 1e-3)
print('--- stage 1: train the new classification head')
histories.append(model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw, callbacks=callbacks(), verbose=2))

base.trainable = True
for layer in base.layers[:int(len(base.layers) * 0.35)]:      # keep the earliest (generic) layers frozen
    layer.trainable = False
for layer in base.layers:                                      # keep BatchNorm statistics frozen
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
compile_model(model, 1e-4)
print('--- stage 2: fine-tune the backbone')
histories.append(model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw, callbacks=callbacks(), verbose=2))

stage2_weights = model.get_weights()
sel_loss, sel_acc, sel_top3 = model.evaluate(val_ds, verbose=0)
print(f'selection-set accuracy after stage 2: top-1 {sel_acc:.4f} | top-3 {sel_top3:.4f}')

## 5. Label-noise pass
Food-101's *training* photos contain wrongly labelled pictures. Hashing cannot find those, but a trained model can: a training image whose label
gets under 5% probability while another class gets over 90% is very likely mislabeled. Those images are listed below (check the gallery),
removed, and the model is fine-tuned a little more. **The cleaned model is kept only if it scores better on the selection set.**

In [ ]:
plain_train = make_ds(train_df, False)
probs = model.predict(plain_train, verbose=0)
pred = probs.argmax(1)
true = train_df['y'].values
p_true = probs[np.arange(len(true)), true]
p_pred = probs.max(1)
flag = (pred != true) & (p_pred >= 0.90) & (p_true <= 0.05)
flagged = train_df[flag].copy()
flagged['predicted'] = [labels[i] for i in pred[flag]]
print(f'{int(flag.sum())} of {len(train_df)} training images look mislabeled ({100 * flag.mean():.1f}%)')
display(flagged['label'].value_counts().to_frame('flagged'))

show = flagged.sample(min(24, len(flagged)), random_state=1) if len(flagged) else flagged
if len(show):
    fig, axes = plt.subplots(4, 6, figsize=(15, 10.5))
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, r) in zip(axes.ravel(), show.iterrows()):
        ax.imshow(plt.imread(r['path'])); ax.set_title(f"labelled: {r['label']}\nmodel: {r['predicted']}", fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'flagged_label_noise.png'), dpi=90); plt.show()

In [ ]:
cleaned_df = train_df[~flag].reset_index(drop=True)
used_cleaned = False
if int(flag.sum()) >= 10:
    print(f'--- fine-tuning on the {len(cleaned_df)} cleaned training images')
    compile_model(model, 3e-5)
    model.fit(make_ds(cleaned_df, True), validation_data=val_ds, epochs=EPOCHS_CLEAN, class_weight=class_weights(cleaned_df),
              callbacks=callbacks(), verbose=2)
    _, acc_clean, top3_clean = model.evaluate(val_ds, verbose=0)
    print(f'selection-set top-1: before {sel_acc:.4f} -> after cleaning {acc_clean:.4f}')
    if acc_clean >= sel_acc:
        used_cleaned = True
        print('=> keeping the model trained on cleaned labels')
    else:
        model.set_weights(stage2_weights)
        print('=> cleaning did not help; restored the stage-2 model')
else:
    print('Too few suspicious images to matter; skipping.')

## 6. Honest evaluation on the held-out test set
These images were never used to train or to make any decision.

In [ ]:
def predict_probs(ds, tta=False):
    p = model.predict(ds, verbose=0)
    if tta:
        flipped = ds.map(lambda x, y: (tf.image.flip_left_right(x), y))
        p = (p + model.predict(flipped, verbose=0)) / 2
    return p

y_test = test_df['y'].values
p_test = predict_probs(test_ds)
p_tta = predict_probs(test_ds, tta=True)
pred_test = p_test.argmax(1)

top1 = float((pred_test == y_test).mean())
top3 = float(np.mean([y in np.argsort(p)[-3:] for p, y in zip(p_test, y_test)]))
top1_tta = float((p_tta.argmax(1) == y_test).mean())
print(f'TEST SET ({len(y_test)} images):  top-1 {top1:.4f} | top-3 {top3:.4f} | top-1 with flip-averaging {top1_tta:.4f}')

by_source = {}
for s_name in sorted(test_df['source'].unique()):
    m = (test_df['source'].values == s_name)
    by_source[s_name] = {'images': int(m.sum()), 'top1': float((pred_test[m] == y_test[m]).mean())}
    print(f"  test top-1 on {s_name}: {by_source[s_name]['top1']:.3f}  ({m.sum()} images)")
print('  (web-scraped sets measure web-photo accuracy, not phone-camera accuracy)')

report = classification_report(y_test, pred_test, target_names=labels, output_dict=True, zero_division=0)
display(pd.DataFrame(report).T.round(3).loc[labels + ['macro avg']])

cm = confusion_matrix(y_test, pred_test)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap='Blues'); ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=75, ha='right'); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=7)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Confusion matrix (held-out test set)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=110); plt.show()

# which confidence threshold should the app use before offering a guess?
conf = p_test.max(1); correct = pred_test == y_test
rows = []
for t in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    keep = conf >= t
    rows.append({'min confidence': t, 'guesses offered': f'{100 * keep.mean():.0f}%', 'correct when offered': f'{100 * correct[keep].mean():.1f}%' if keep.any() else '-'})
display(pd.DataFrame(rows))
recommended = next((r['min confidence'] for r in rows if r['correct when offered'] != '-' and float(r['correct when offered'][:-1]) >= 90), 0.5)
print('recommended minimum confidence for the app:', recommended)

hist = {k: sum((h.history.get(k, []) for h in histories), []) for k in ['accuracy', 'val_accuracy', 'loss', 'val_loss']}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(hist['accuracy'], label='train'); ax[0].plot(hist['val_accuracy'], label='selection'); ax[0].set_title('accuracy'); ax[0].legend()
ax[1].plot(hist['loss'], label='train'); ax[1].plot(hist['val_loss'], label='selection'); ax[1].set_title('loss'); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=100); plt.show()

## 7. Export to TensorFlow Lite and verify it
Float16 weights halve the file size with no practical accuracy change. The exported model is run through the real TFLite interpreter and must
**give the same answers as the trained model** before it can be shipped.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()
candidate_path = os.path.join(OUT_DIR, 'food_classifier_CANDIDATE.tflite')
open(candidate_path, 'wb').write(tflite_bytes)
print('TFLite size: %.1f MB' % (len(tflite_bytes) / 1e6))

interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inp, outp = interp.get_input_details()[0], interp.get_output_details()[0]
n_check = min(len(test_df), 500)
agree = tflite_hits = 0
max_diff = 0.0
for k, (x, y) in enumerate(test_ds.unbatch().take(n_check)):
    interp.set_tensor(inp['index'], x.numpy()[None].astype(np.float32))
    interp.invoke()
    p_lite = interp.get_tensor(outp['index'])[0]
    p_keras = p_test[k]
    agree += int(p_lite.argmax() == p_keras.argmax())
    tflite_hits += int(p_lite.argmax() == int(y))
    max_diff = max(max_diff, float(np.abs(p_lite - p_keras).max()))
parity = agree / n_check
tflite_acc = tflite_hits / n_check
print(f'TFLite vs trained model on {n_check} test images: same answer {100 * parity:.1f}% | TFLite top-1 {tflite_acc:.4f} | largest probability difference {max_diff:.4f}')

## 8. Shipping gate
The model is written as `food_classifier.tflite` **only if all checks pass**; otherwise it is saved as `food_classifier_CANDIDATE.tflite`
and the notebook says exactly why. Thresholds are deliberately strict: a wrong food guess means wrong calories in someone's diary.

In [ ]:
MIN_TOP1, MIN_CLASS_RECALL, MIN_PARITY = 0.85, 0.70, 0.98
worst_class = min(labels, key=lambda l: report[l]['recall'])
checks = {
    f'test top-1 >= {MIN_TOP1:.0%}': top1 >= MIN_TOP1,
    f'every class recall >= {MIN_CLASS_RECALL:.0%} (worst: {worst_class} {report[worst_class]["recall"]:.0%})': report[worst_class]['recall'] >= MIN_CLASS_RECALL,
    f'TFLite agrees with the trained model on >= {MIN_PARITY:.0%} of images': parity >= MIN_PARITY,
}
for name, ok in checks.items():
    print(('PASS  ' if ok else 'FAIL  ') + name)
ship = all(checks.values()) and not QUICK

labels_json = {
    'name': 'food_classifier', 'version': datetime.now(timezone.utc).strftime('%Y%m%d'),
    'trained_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'architecture': f'{BACKBONE} (transfer learning, fine-tuned), float16 TFLite',
    'input': {'size': IMG, 'channels': 'RGB', 'dtype': 'float32', 'range': '0-255 (preprocessing is inside the model)', 'resize': 'area/average, squashed to square'},
    'labels': labels,
    'recommended_min_confidence': recommended,
    'ship_recommended': bool(ship),
    'metrics': {'test_images': int(len(y_test)), 'test_top1': top1, 'test_top3': top3, 'test_top1_flip_avg': top1_tta,
                'tflite_top1_sample': tflite_acc, 'test_top1_by_source': by_source, 'tflite_keras_agreement': parity,
                'per_class': {l: {k: round(report[l][k], 4) for k in ['precision', 'recall', 'f1-score']} for l in labels}},
    'training': {'train_images': int(len(train_df)), 'label_noise_images_removed': int(flag.sum()) if used_cleaned else 0,
                 'dataset': 'Food-101 subset (cleaned), terms of Food-101 apply; sources in this run: ' + ', '.join(sorted(meta['source'].unique()))
                            + (' + the team\'s own photos (' + ', '.join(sorted(l for l, r in custom_report.items() if r['kept'])) + ')'
                               if any(r['kept'] for r in custom_report.values()) else ''),
                 'custom_photo_dishes_not_trained': {l: r['problems'] for l, r in custom_report.items() if not r['kept']}},
}
with open(os.path.join(OUT_DIR, 'food_classifier_labels.json'), 'w') as f:
    json.dump(labels_json, f, indent=1)

if ship:
    os.replace(candidate_path, os.path.join(OUT_DIR, 'food_classifier.tflite'))
    print('\nSHIP: wrote food_classifier.tflite + food_classifier_labels.json')
    print('Copy BOTH files into the app folder  assets/models/  and rebuild the app.')
else:
    print('\nDO NOT SHIP yet. The model is saved as food_classifier_CANDIDATE.tflite for inspection only.')
    if QUICK:
        print('(quick smoke-test mode: this model is untrained by design)')
print('files in', OUT_DIR, ':', sorted(os.listdir(OUT_DIR)))

## 9. After training
* Copy `food_classifier.tflite` and `food_classifier_labels.json` into the app's `assets/models/` folder and rebuild. The app reads the input size and the
  recommended confidence from the JSON file.
* Look at `confusion_matrix.png`: classes that are confused with each other (karahi vs nihari vs curries, pasta, the three cake types) show where more or better photos are needed.
* **Pakistani dishes** come only from your own photos (`custom_photos/`, see 1a). Dishes listed as *NOT TRAINED* above need more photos from **more
  different sessions/plates**; the app keeps manual selection for them. Add photos and re-run.
* The model recognises the dish, not the portion size; the app lets the user adjust the portion.